# 00 · Core: shared code and its tests

Every part notebook starts with `%run ./00_core.ipynb`. Data loading, ground truth, recall@10, the timing harness, index building and the results writer therefore exist exactly once, and every part measures the same way. Each feature cell is followed by its pytest cell, run with `ipytest`; a failing test raises and stops `run_all.sh`.

The mode comes from `ANN_MODE`: `full`, or `quick` (the default, for interactive use). `run_all.sh` sets it. Measurement choices are explained in `design_note.md`.

In [ ]:
import contextlib
import dataclasses
import hashlib
import json
import os
import platform
import subprocess
import time
from pathlib import Path

import faiss
import h5py
import ipytest
import numpy as np
import pandas as pd
import pytest
import threadpoolctl

ipytest.autoconfig(raise_on_error=True)

## Configuration
Every size, grid and query range for both modes lives here, so `--quick` runs exactly the same code at a smaller scale.

In [ ]:
def find_root():
    if "ANN_ROOT" in os.environ:
        return Path(os.environ["ANN_ROOT"]).resolve()
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "run_all.sh").exists():
            return path
    raise FileNotFoundError("no run_all.sh in the current directory or above it")


MODE = os.environ.get("ANN_MODE", "quick")
assert MODE in ("full", "quick"), f"ANN_MODE must be 'full' or 'quick', not {MODE!r}"
ROOT = find_root()
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "cache"
RESULTS_DIR = ROOT / ("results" if MODE == "full" else "results_quick")
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = ROOT / "report" / "tables" if MODE == "full" else RESULTS_DIR / "tables"
for directory in (CACHE_DIR, FIGURES_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
K = 10

_SHARED = dict(
    repeats=3,           # timed passes per setting; the median is reported
    warmup=200,          # untimed queries before each timed setting
    recall_stop=0.995,   # a sweep stops once tune-half recall reaches this
    gt_k=100,            # depth of the cached ground truth
    hnsw_efC=200,
    scaling_M=16,
    noise_M=16,
    noise_sigmas=[0.0, 0.1, 0.3, 1.0],  # 0.0 is the unchanged control
)
FULL = dict(
    _SHARED,
    n_base=None,                        # every base vector
    tune=(0, 5_000),
    report=(5_000, 10_000),
    exact_timing=(5_000, 6_000),        # report queries used to time exact search one at a time
    agreement_queries=10_000,           # queries checked against the stored neighbours (Part 1)
    hnsw_M=[8, 16, 32],
    ef_search=[10, 16, 32, 64, 128, 256, 512],
    ivf_nlist=[1024, 4096],
    nprobe=[1, 2, 4, 8, 16, 32, 64, 128, 256],
    ivfpq_nlist=4096,
    ivfpq_m={"sift": [16, 32, 64], "glove": [20, 50]},
    scaling_sizes=[100_000, 300_000, 1_000_000],
    noise_n_base=200_000,
    noise_nlist=1024,
)
QUICK = dict(
    _SHARED,
    n_base=100_000,                     # the first 100,000 base vectors
    tune=(0, 1_000),
    report=(5_000, 6_000),
    exact_timing=(5_000, 6_000),
    agreement_queries=1_000,
    hnsw_M=[16],
    ef_search=[10, 16, 32, 64, 128, 256],
    ivf_nlist=[256],
    nprobe=[1, 2, 4, 8, 16, 32, 64, 128],
    ivfpq_nlist=256,
    ivfpq_m={"sift": [32], "glove": [20]},
    scaling_sizes=[10_000, 30_000, 100_000],
    noise_n_base=50_000,
    noise_nlist=256,
)
CONFIG = FULL if MODE == "full" else QUICK

In [ ]:
%%ipytest -qq

@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_query_ranges_follow_the_assignment_split(config):
    (t0, t1), (r0, r1), (e0, e1) = config["tune"], config["report"], config["exact_timing"]
    assert 0 <= t0 < t1 <= 5_000          # tune only on the first 5,000 queries
    assert 5_000 <= r0 < r1 <= 10_000     # report only on the last 5,000
    assert r0 <= e0 < e1 <= r1            # exact timing uses report queries


@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_pq_sub_vectors_divide_the_dimension(config):
    for name, dim in (("sift", 128), ("glove", 100)):
        assert all(dim % m == 0 for m in config["ivfpq_m"][name])


@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_grids_increase_and_part4_follows_its_rules(config):
    for key in ("ef_search", "nprobe", "scaling_sizes", "noise_sigmas"):
        assert config[key] == sorted(config[key])
    assert config["noise_sigmas"][0] == 0.0 and len(config["noise_sigmas"]) >= 4  # control + 3 settings
    assert config["noise_n_base"] <= 500_000                                       # at most 500K vectors

## Machine record
Stored in every results row, so each number carries the machine and library versions it came from. The timing thread is pinned to one performance core (CPU 2 here; see `design_note.md`).

In [ ]:
def parse_cpu_list(text):
    """'0-3,8,10-11' -> [0, 1, 2, 3, 8, 10, 11]"""
    cpus = []
    for part in text.strip().split(","):
        if part:
            low, _, high = part.partition("-")
            cpus.extend(range(int(low), int(high or low) + 1))
    return cpus


def _read(path, default=""):
    try:
        return Path(path).read_text().strip()
    except OSError:
        return default


def performance_cores():
    """CPUs of the performance cores on a hybrid Intel CPU; [] where the kernel doesn't report them."""
    return parse_cpu_list(_read("/sys/devices/cpu_core/cpus"))


def default_timing_cpu():
    allowed = sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else []
    candidates = [cpu for cpu in performance_cores() if cpu in allowed] or allowed
    # The third CPU in the list (CPU 2 here): a P-core thread, away from CPU 0, which takes more interrupts.
    return candidates[min(2, len(candidates) - 1)] if candidates else None


TIMING_CPU = int(os.environ["ANN_TIMING_CPU"]) if "ANN_TIMING_CPU" in os.environ else default_timing_cpu()
BUILD_THREADS = len(performance_cores()) or os.cpu_count()


def machine_info():
    cpuinfo = _read("/proc/cpuinfo").splitlines()
    meminfo = _read("/proc/meminfo").splitlines()
    cpu_model = next((line.split(":", 1)[1].strip() for line in cpuinfo if line.startswith("model name")), platform.processor())
    mem_kb = next((int(line.split()[1]) for line in meminfo if line.startswith("MemTotal")), 0)
    return dict(
        cpu_model=cpu_model,
        logical_cpus=os.cpu_count(),
        ram_gb=round(mem_kb / 2**20, 1),
        os=platform.platform(),
        python=platform.python_version(),
        numpy=np.__version__,
        faiss=faiss.__version__,
        governor=_read(f"/sys/devices/system/cpu/cpu{TIMING_CPU or 0}/cpufreq/scaling_governor", "unknown"),
        timing_cpu=TIMING_CPU,
        build_threads=BUILD_THREADS,
    )


MACHINE = machine_info()
MACHINE

In [ ]:
%%ipytest -qq

def test_parse_cpu_list():
    assert parse_cpu_list("0-3,8,10-11") == [0, 1, 2, 3, 8, 10, 11]
    assert parse_cpu_list("5") == [5]
    assert parse_cpu_list("") == []


def test_machine_record_is_complete():
    assert set(MACHINE) >= {"cpu_model", "ram_gb", "os", "python", "numpy", "faiss", "governor", "timing_cpu"}
    assert MACHINE["ram_gb"] > 0 and BUILD_THREADS >= 1


@pytest.mark.skipif(not hasattr(os, "sched_getaffinity"), reason="no CPU affinity on this platform")
def test_timing_cpu_is_one_we_may_run_on():
    assert TIMING_CPU in os.sched_getaffinity(0)

## Data
SIFT and GloVe come from the ann-benchmarks HDF5 files. GloVe base vectors and queries are scaled to unit length, so Euclidean distance ranks them exactly as angular distance does. The stored neighbours are kept only when the full base set is loaded, because they are wrong for any subset. Queries 0–4,999 are for tuning and 5,000–9,999 for reported numbers (quick mode uses the first 1,000 of each).

In [ ]:
DATASETS = {
    "sift": dict(file="sift-128-euclidean.hdf5", normalise=False),
    "glove": dict(file="glove-100-angular.hdf5", normalise=True),
}


@dataclasses.dataclass
class Dataset:
    name: str
    base: np.ndarray                     # (n, d) float32, C-contiguous
    queries: np.ndarray                  # (10_000, d) float32
    stored_neighbors: np.ndarray | None  # HDF5 ground truth; None unless the full base set is loaded
    stored_distances: np.ndarray | None

    @property
    def dim(self):
        return self.base.shape[1]


def normalise(x):
    """Scale rows to unit L2 norm; all-zero rows stay zero instead of becoming NaN."""
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return np.divide(x, norms, out=np.zeros_like(x), where=norms > 0)


def load_dataset(name, n_base=None, data_dir=None):
    """Base vectors (the first n_base, or all of them) and the 10,000 queries, as float32."""
    spec = DATASETS[name]
    with h5py.File(Path(data_dir or DATA_DIR) / spec["file"], "r") as f:
        full = n_base is None or n_base >= f["train"].shape[0]
        base = f["train"][:] if full else f["train"][:n_base]
        queries = f["test"][:]
        neighbors = f["neighbors"][:] if full else None
        distances = f["distances"][:] if full else None
    base = np.ascontiguousarray(base, dtype=np.float32)
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    if spec["normalise"]:
        base, queries = normalise(base), normalise(queries)
    return Dataset(name, base, queries, neighbors, distances)


TUNE = np.arange(*CONFIG["tune"])
REPORT = np.arange(*CONFIG["report"])

In [ ]:
%%ipytest -qq

def write_toy_hdf5(path, n=50, d=4, nq=10):
    rng = np.random.default_rng(1)
    with h5py.File(path, "w") as f:
        f["train"] = rng.random((n, d))
        f["test"] = rng.random((nq, d))
        f["neighbors"] = np.zeros((nq, 5), dtype=np.int32)
        f["distances"] = np.zeros((nq, 5))


def test_normalise_gives_unit_rows_and_keeps_zero_rows():
    y = normalise(np.array([[3, 4], [0, 0], [1e-3, 0]], dtype=np.float32))
    assert y.dtype == np.float32 and not np.isnan(y).any()
    assert np.allclose(np.linalg.norm(y[[0, 2]], axis=1), 1)
    assert (y[1] == 0).all()


def test_load_dataset_subsets_and_normalises(tmp_path):
    write_toy_hdf5(tmp_path / "glove-100-angular.hdf5")
    full = load_dataset("glove", data_dir=tmp_path)
    part = load_dataset("glove", n_base=20, data_dir=tmp_path)
    assert full.base.dtype == np.float32 and full.base.flags.c_contiguous
    assert np.allclose(np.linalg.norm(full.base, axis=1), 1)
    assert part.base.shape == (20, 4) and (part.base == full.base[:20]).all()
    assert full.stored_neighbors is not None and part.stored_neighbors is None


def test_query_halves_are_disjoint_and_on_the_right_side():
    assert set(TUNE).isdisjoint(REPORT)
    assert TUNE.max() < 5_000 <= REPORT.min() and REPORT.max() < 10_000

## Ground truth
This is our own exact search: a batched `IndexFlatL2` run with all build threads. It is never timed, and it is cached in `cache/`. The cache file name includes a fingerprint of the data, so a different subset or noise level never reuses the wrong file.

In [ ]:
@contextlib.contextmanager
def faiss_threads(n):
    old = faiss.omp_get_max_threads()
    faiss.omp_set_num_threads(n)
    try:
        yield
    finally:
        faiss.omp_set_num_threads(old)


def exact_knn(base, queries, k):
    """Exact k nearest neighbours by squared L2 distance (batched, multi-threaded, untimed)."""
    index = faiss.IndexFlatL2(base.shape[1])
    index.add(base)
    with faiss_threads(BUILD_THREADS):
        return index.search(queries, k)


def fingerprint(*arrays):
    digest = hashlib.sha1()
    for a in arrays:
        digest.update(str(a.shape).encode())
        digest.update(np.ascontiguousarray(a[:: max(1, len(a) // 1000)]).tobytes())
    return digest.hexdigest()[:12]


def ground_truth(tag, base, queries, k=None, cache_dir=None):
    """Exact top-k ids for every query, cached under a name tied to the data itself."""
    k = k or CONFIG["gt_k"]
    path = Path(cache_dir or CACHE_DIR) / f"gt_{tag}_{len(base)}x{base.shape[1]}_q{len(queries)}_k{k}_{fingerprint(base, queries)}.npy"
    if path.exists():
        return np.load(path)
    ids = exact_knn(base, queries, k)[1]
    np.save(path, ids)
    return ids

In [ ]:
%%ipytest -qq

def random_data(n=300, d=8, nq=20, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n, d), dtype=np.float32), rng.random((nq, d), dtype=np.float32)


def test_exact_knn_matches_numpy_brute_force():
    base, queries = random_data()
    distances, ids = exact_knn(base, queries, 10)
    true = ((queries[:, None, :].astype(np.float64) - base[None]) ** 2).sum(-1)
    assert (ids == np.argsort(true, axis=1)[:, :10]).all()
    assert np.allclose(distances, np.sort(true, axis=1)[:, :10], rtol=1e-4)


def test_ground_truth_is_cached_and_keyed_by_the_data(tmp_path):
    base, queries = random_data()
    first = ground_truth("toy", base, queries, k=5, cache_dir=tmp_path)
    assert len(list(tmp_path.glob("gt_toy_*.npy"))) == 1
    assert (ground_truth("toy", base, queries, k=5, cache_dir=tmp_path) == first).all()
    ground_truth("toy", base + 1, queries, k=5, cache_dir=tmp_path)  # different data, so a new file
    assert len(list(tmp_path.glob("gt_toy_*.npy"))) == 2

## Recall@10
A returned id counts as correct if its distance to the query is at most the distance of the true 10th neighbour. Ties at rank 10 are therefore not mistakes. Both distances are recomputed in float64 from the stored vectors, with a small relative tolerance. A FAISS `-1` (no result) is a miss; used as a numpy index it would silently pick the last vector. An id returned twice counts once.

In [ ]:
def recall_at_k(found, base, queries, gt_ids, k=K, rtol=1e-6, per_query=False):
    """Tie-aware recall@k: the share of returned ids that are no farther than the true k-th neighbour."""
    found = np.asarray(found)[:, :k]
    q = queries.astype(np.float64)
    kth = base[gt_ids[:, k - 1]].astype(np.float64)
    threshold = ((kth - q) ** 2).sum(axis=1) * (1 + rtol)

    valid = found >= 0
    vectors = base[np.where(valid, found, 0)].astype(np.float64)
    distance = ((vectors - q[:, None, :]) ** 2).sum(axis=2)

    # Flag every repeat of an id after its first appearance in the row, so duplicates count once.
    order = np.argsort(found, axis=1, kind="stable")
    ids_sorted = np.take_along_axis(found, order, axis=1)
    repeat_sorted = np.zeros_like(valid)
    repeat_sorted[:, 1:] = ids_sorted[:, 1:] == ids_sorted[:, :-1]
    repeat = np.zeros_like(valid)
    np.put_along_axis(repeat, order, repeat_sorted, axis=1)

    correct = valid & ~repeat & (distance <= threshold[:, None])
    scores = correct.sum(axis=1) / k
    return scores if per_query else float(scores.mean())

In [ ]:
%%ipytest -qq

def line_data():
    """A query at the origin and 20 points on the x-axis at distances 1..20: the true top 10 is ids 0..9."""
    base = np.zeros((20, 2), dtype=np.float32)
    base[:, 0] = np.arange(1, 21)
    return base, np.zeros((1, 2), dtype=np.float32), np.arange(20)[None, :]


def with_point(base, point):
    return np.vstack([base, [point]]).astype(np.float32)


def test_perfect_result_scores_one():
    base, q, gt = line_data()
    assert recall_at_k(gt[:, :10], base, q, gt) == 1.0


def test_wrong_result_scores_zero():
    base, q, gt = line_data()
    assert recall_at_k(np.arange(10, 20)[None, :], base, q, gt) == 0.0


def test_partial_result():
    base, q, gt = line_data()
    assert recall_at_k(np.array([[0, 1, 2, 3, 4, 5, 6, 15, 16, 17]]), base, q, gt) == pytest.approx(0.7)


def test_tie_at_the_tenth_neighbour_is_not_a_mistake():
    base, q, gt = line_data()
    base = with_point(base, [0, 10])  # id 20 is exactly as far away as the 10th neighbour (id 9)
    assert recall_at_k(np.array([[0, 1, 2, 3, 4, 5, 6, 7, 8, 20]]), base, q, gt) == 1.0


def test_point_just_beyond_the_tenth_neighbour_is_a_mistake():
    base, q, gt = line_data()
    base = with_point(base, [0, 10.01])
    assert recall_at_k(np.array([[0, 1, 2, 3, 4, 5, 6, 7, 8, 20]]), base, q, gt) == pytest.approx(0.9)


def test_minus_one_is_a_miss_not_the_last_vector():
    base, q, _ = line_data()
    base = with_point(base, [0.5, 0])        # the last vector (id 20) is now the nearest neighbour
    gt = np.array([[20, *range(19)]])
    found = np.array([[0, 1, 2, 3, 4, 5, 6, 7, -1, -1]])
    assert recall_at_k(found, base, q, gt) == pytest.approx(0.8)


def test_duplicate_ids_count_once():
    base, q, gt = line_data()
    assert recall_at_k(np.zeros((1, 10), dtype=np.int64), base, q, gt) == pytest.approx(0.1)


def test_order_does_not_matter():
    base, q, gt = line_data()
    found = np.array([[9, 3, 15, 0, 7, 12, 1, 5, 2, 8]])
    assert recall_at_k(found, base, q, gt) == recall_at_k(np.sort(found), base, q, gt) == pytest.approx(0.8)


def test_mean_and_per_query_scores():
    base, q, gt = line_data()
    found = np.vstack([gt[:, :10], np.arange(10, 20)[None, :]])
    queries, gts = np.vstack([q, q]), np.vstack([gt, gt])
    assert recall_at_k(found, base, queries, gts) == pytest.approx(0.5)
    assert recall_at_k(found, base, queries, gts, per_query=True).tolist() == [1.0, 0.0]

## Timing
QPS is measured one query at a time on a single thread: one FAISS thread, one BLAS thread, and the calling thread pinned to `TIMING_CPU`. Queries are sliced into one-row float32 views before the clock starts. Each setting gets untimed warm-up queries, then several timed passes, and the median is reported. Batch timing (Part 1 only) runs either the same pinned single thread or `n` unpinned threads.

In [ ]:
@contextlib.contextmanager
def single_thread_pinned(cpu=None):
    cpu = TIMING_CPU if cpu is None else cpu
    can_pin = hasattr(os, "sched_setaffinity") and cpu is not None
    old_affinity = os.sched_getaffinity(0) if can_pin else None
    with faiss_threads(1), threadpoolctl.threadpool_limits(1):
        if can_pin:
            os.sched_setaffinity(0, {cpu})
        try:
            yield
        finally:
            if can_pin:
                os.sched_setaffinity(0, old_affinity)


@contextlib.contextmanager
def search_threads(n):
    """n == 1: the pinned single-thread setup used for every QPS number; n > 1: n FAISS/BLAS threads, unpinned."""
    if n == 1:
        with single_thread_pinned():
            yield
    else:
        with faiss_threads(n), threadpoolctl.threadpool_limits(n):
            yield


def time_one_at_a_time(index, queries, k=K, repeats=None, warmup=None):
    """Median QPS of index.search called once per query, every pass's time, and the ids from the last pass."""
    repeats = repeats or CONFIG["repeats"]
    warmup = CONFIG["warmup"] if warmup is None else warmup
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    rows = [queries[i : i + 1] for i in range(len(queries))]
    ids = np.empty((len(rows), k), dtype=np.int64)
    id_rows = [ids[i : i + 1] for i in range(len(rows))]
    distances = np.empty((1, k), dtype=np.float32)
    times = []
    with search_threads(1):
        for i in range(min(warmup, len(rows))):
            index.search(rows[i], k, D=distances, I=id_rows[i])
        for _ in range(repeats):
            start = time.perf_counter()
            for row, id_row in zip(rows, id_rows):
                index.search(row, k, D=distances, I=id_row)
            times.append(time.perf_counter() - start)
    return dict(qps=len(rows) / float(np.median(times)), times=times, ids=ids)


def time_batch(index, queries, k=K, threads=1, repeats=None):
    """Median QPS of a single search call over all queries with `threads` threads."""
    repeats = repeats or CONFIG["repeats"]
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    times = []
    with search_threads(threads):
        index.search(queries[: CONFIG["warmup"]], k)
        for _ in range(repeats):
            start = time.perf_counter()
            index.search(queries, k)
            times.append(time.perf_counter() - start)
    return dict(qps=len(queries) / float(np.median(times)), times=times)

In [ ]:
%%ipytest -qq

class CountingIndex:
    """Stands in for a FAISS index: counts search calls and returns ids 0..k-1."""

    def __init__(self):
        self.calls = 0

    def search(self, x, k, D=None, I=None):
        self.calls += 1
        I = np.empty((len(x), k), dtype=np.int64) if I is None else I
        I[:] = np.arange(k)
        return D, I


def test_one_at_a_time_counts_warmup_and_repeats():
    index, queries = CountingIndex(), np.zeros((50, 4), dtype=np.float32)
    result = time_one_at_a_time(index, queries, repeats=3, warmup=5)
    assert index.calls == 5 + 3 * 50
    assert len(result["times"]) == 3 and result["ids"].shape == (50, K)
    assert result["qps"] == pytest.approx(50 / np.median(result["times"]))


def test_batch_timing_is_one_call_per_pass():
    index = CountingIndex()
    result = time_batch(index, np.zeros((50, 4), dtype=np.float32), repeats=3)
    assert index.calls == 1 + 3 and len(result["times"]) == 3


@pytest.mark.skipif(not hasattr(os, "sched_getaffinity"), reason="no CPU affinity on this platform")
def test_single_thread_pinning_is_undone_afterwards():
    threads_before, cpus_before = faiss.omp_get_max_threads(), os.sched_getaffinity(0)
    with single_thread_pinned():
        assert faiss.omp_get_max_threads() == 1
        assert os.sched_getaffinity(0) == {TIMING_CPU}
    assert faiss.omp_get_max_threads() == threads_before
    assert os.sched_getaffinity(0) == cpus_before


def test_one_at_a_time_ids_match_a_batch_search():
    base, queries = random_data()
    index = faiss.IndexFlatL2(base.shape[1])
    index.add(base)
    result = time_one_at_a_time(index, queries, repeats=1, warmup=0)
    assert (result["ids"] == index.search(queries, K)[1]).all()

## Indexes
All four indexes come from FAISS. k-means seeds are fixed. Build time is the wall-clock time of train + add with `BUILD_THREADS` threads. Index size is the size of the serialised index, which is held in memory and never written to disk.

In [ ]:
SEARCH_PARAM = {"exact": "", "hnsw": "efSearch", "ivf_flat": "nprobe", "ivf_pq": "nprobe"}


def make_index(kind, dim, **params):
    """An empty FAISS index: 'exact', 'hnsw' (M, efC), 'ivf_flat' (nlist) or 'ivf_pq' (nlist, m)."""
    if kind == "exact":
        return faiss.IndexFlatL2(dim)
    if kind == "hnsw":
        index = faiss.index_factory(dim, f"HNSW{params['M']}")
        index.hnsw.efConstruction = params["efC"]
        return index
    if kind == "ivf_flat":
        index = faiss.index_factory(dim, f"IVF{params['nlist']},Flat")
    elif kind == "ivf_pq":
        index = faiss.index_factory(dim, f"IVF{params['nlist']},PQ{params['m']}")
        index.pq.cp.seed = SEED
    else:
        raise ValueError(f"unknown index kind {kind!r}")
    index.cp.seed = SEED
    return index


def set_search_param(index, kind, value):
    if kind == "hnsw":
        index.hnsw.efSearch = int(value)
    elif kind in ("ivf_flat", "ivf_pq"):
        index.nprobe = int(value)


def build_index(kind, base, **params):
    """Train (if needed) and add `base`; returns the index and its wall-clock build time in seconds."""
    index = make_index(kind, base.shape[1], **params)
    with faiss_threads(BUILD_THREADS), threadpoolctl.threadpool_limits(BUILD_THREADS):
        start = time.perf_counter()
        if not index.is_trained:
            index.train(base)
        index.add(base)
        build_time = time.perf_counter() - start
    return index, build_time


def index_bytes(index):
    return int(faiss.serialize_index(index).nbytes)

In [ ]:
%%ipytest -qq

@pytest.mark.parametrize("kind, params", [
    ("exact", {}), ("hnsw", dict(M=8, efC=40)), ("ivf_flat", dict(nlist=8)), ("ivf_pq", dict(nlist=8, m=4)),
])
def test_every_index_kind_builds_and_searches(kind, params):
    base, queries = random_data(n=10_000, d=8)
    index, build_time = build_index(kind, base, **params)
    set_search_param(index, kind, 16)
    _, ids = index.search(queries, K)
    assert index.ntotal == len(base) and build_time > 0 and index_bytes(index) > 0
    assert ids.shape == (len(queries), K) and (ids >= 0).all()


def test_kmeans_seeds_are_fixed():
    assert make_index("ivf_flat", 8, nlist=8).cp.seed == SEED
    pq = make_index("ivf_pq", 8, nlist=8, m=4)
    assert pq.cp.seed == SEED and pq.pq.cp.seed == SEED


def test_search_parameters_are_applied():
    hnsw = make_index("hnsw", 8, M=8, efC=40)
    set_search_param(hnsw, "hnsw", 77)
    ivf = make_index("ivf_flat", 8, nlist=8)
    set_search_param(ivf, "ivf_flat", 5)
    assert hnsw.hnsw.efSearch == 77 and ivf.nprobe == 5


def test_index_size_grows_with_the_number_of_vectors():
    base, _ = random_data(n=2_000)
    assert index_bytes(build_index("exact", base)[0]) > index_bytes(build_index("exact", base[:500])[0])

## Benchmark runs
`benchmark_index` builds one configuration, records its build cost, then sweeps the search parameter. Every setting is measured on both query halves, and the sweep stops once the tune-half recall reaches `recall_stop`. `benchmark_exact` times exact search on the exact-timing queries. Every row carries the run's settings and the machine record.

In [ ]:
def run_header(part, section, data, kind, params, build_time, size, variant=""):
    return dict(
        part=part, section=section, dataset=data.name, variant=variant, n_base=len(data.base), dim=data.dim,
        index=kind, build_params=json.dumps(params, sort_keys=True), build_time_s=round(build_time, 4),
        index_bytes=size, bytes_per_vector=size / len(data.base), search_threads=1,
        mode=MODE, seed=SEED, timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"), **MACHINE,
    )


def measure(index, data, gt, split, rows=None):
    """Recall@10 and single-thread QPS on one query half ('tune' or 'report'), or on explicit query rows."""
    rows = (TUNE if split == "tune" else REPORT) if rows is None else rows
    queries = data.queries[rows]
    timing = time_one_at_a_time(index, queries)
    return dict(
        split=split, n_queries=len(rows),
        recall_at_10=recall_at_k(timing["ids"], data.base, queries, gt[rows]),
        qps=timing["qps"], times_s=json.dumps([round(t, 6) for t in timing["times"]]),
    )


def run_sweep(index, kind, values, data, gt, header, halves=None):
    halves = halves or {"tune": TUNE, "report": REPORT}
    rows = []
    for value in values:
        set_search_param(index, kind, value)
        for split, query_rows in halves.items():
            rows.append({**header, "param_name": SEARCH_PARAM[kind], "param_value": value,
                         **measure(index, data, gt, split, query_rows)})
        tune_recall = next(r["recall_at_10"] for r in rows[::-1] if r["split"] == "tune")
        if tune_recall >= CONFIG["recall_stop"]:
            break
    return rows


def benchmark_index(part, section, data, gt, kind, params, values, variant="", halves=None):
    """Build one configuration, record its build time and size, then sweep its search parameter."""
    index, build_time = build_index(kind, data.base, **params)
    header = run_header(part, section, data, kind, params, build_time, index_bytes(index), variant)
    rows = run_sweep(index, kind, values, data, gt, header, halves)
    best = max(r["recall_at_10"] for r in rows if r["split"] == "tune")
    print(f"{section} {variant} {kind} {params}: built in {build_time:.1f} s, best tune recall {best:.3f}")
    return rows


def benchmark_exact(part, section, data, gt, variant="", rows=None):
    """Exact search timed one query at a time on the exact-timing queries; recall is 1 by construction."""
    index, build_time = build_index("exact", data.base)
    header = run_header(part, section, data, "exact", {}, build_time, index_bytes(index), variant)
    rows = np.arange(*CONFIG["exact_timing"]) if rows is None else rows
    return [{**header, "param_name": "", "param_value": 0, **measure(index, data, gt, "report", rows)}]

In [ ]:
%%ipytest -qq

def toy_dataset(n=3_000, nq=60):
    base, queries = random_data(n=n, d=8, nq=nq)
    return Dataset("toy", base, queries, None, None), exact_knn(base, queries, 20)[1]


def test_sweep_measures_both_halves_and_stops_once_tune_recall_is_high():
    data, gt = toy_dataset()
    halves = {"tune": np.arange(30), "report": np.arange(30, 60)}
    rows = benchmark_index(0, "test", data, gt, "hnsw", dict(M=8, efC=40), [10, 16, 32, 64, 128, 256], halves=halves)
    assert [r["split"] for r in rows] == ["tune", "report"] * (len(rows) // 2)
    tune = [r["recall_at_10"] for r in rows if r["split"] == "tune"]
    assert all(r < CONFIG["recall_stop"] for r in tune[:-1])
    assert all(0 <= r <= 1 for r in tune) and rows[0]["param_name"] == "efSearch"


def test_exact_search_has_perfect_recall():
    data, gt = toy_dataset()
    [row] = benchmark_exact(0, "test", data, gt, rows=np.arange(30, 60))
    assert row["recall_at_10"] == 1.0 and row["index"] == "exact" and row["split"] == "report"

## Choosing configurations (tune half only)
Pareto fronts and operating points are picked from **tune-half** rows only. The reported numbers are the matching **report-half** rows, even when those land just under a target.

In [ ]:
CONFIG_KEYS = ["part", "section", "dataset", "variant", "n_base", "index", "build_params", "param_name", "param_value"]


def pareto_front(rows):
    """Tune-half rows that no other tune-half row beats on both recall and QPS, sorted by recall."""
    tune = rows[rows["split"] == "tune"].sort_values(["recall_at_10", "qps"], ascending=[False, False])
    keep, best_qps = [], -np.inf
    for label, row in tune.iterrows():
        if row["qps"] > best_qps:
            keep.append(label)
            best_qps = row["qps"]
    return tune.loc[keep].sort_values("recall_at_10")


def report_rows_for(chosen, rows):
    """The report-half rows measured with the same configurations as `chosen` (tune-half rows)."""
    report = rows[rows["split"] == "report"]
    return chosen[CONFIG_KEYS].merge(report, on=CONFIG_KEYS, how="left")


def pick_operating_point(rows, target):
    """The fastest configuration whose tune-half recall reaches `target`, as its report-half row (None if none does)."""
    reached = rows[(rows["split"] == "tune") & (rows["recall_at_10"] >= target)]
    if reached.empty:
        return None
    return report_rows_for(reached.loc[[reached["qps"].idxmax()]], rows).iloc[0]

In [ ]:
%%ipytest -qq

def toy_rows():
    """One index, five efSearch values: (tune recall, tune QPS, report recall, report QPS)."""
    common = dict(part=2, section="toy", dataset="toy", variant="", n_base=100, index="hnsw",
                  build_params="{}", param_name="efSearch")
    measured = {16: (0.80, 1000, 0.79, 990), 32: (0.92, 600, 0.90, 610), 48: (0.91, 500, 0.95, 490),
                64: (0.96, 300, 0.94, 305), 128: (0.99, 150, 0.99, 140)}
    rows = []
    for value, (tr, tq, rr, rq) in measured.items():
        rows.append(dict(common, param_value=value, split="tune", recall_at_10=tr, qps=tq))
        rows.append(dict(common, param_value=value, split="report", recall_at_10=rr, qps=rq))
    return pd.DataFrame(rows)


def test_pareto_front_drops_dominated_settings():
    assert pareto_front(toy_rows())["param_value"].tolist() == [16, 32, 64, 128]  # 48 is beaten by 32


def test_operating_point_is_chosen_on_the_tune_half_and_reported_on_the_report_half():
    at_090 = pick_operating_point(toy_rows(), 0.90)
    assert at_090["param_value"] == 32 and at_090["recall_at_10"] == 0.90 and at_090["qps"] == 610
    at_095 = pick_operating_point(toy_rows(), 0.95)
    # The report half alone would suggest 48 (0.95 at 490 QPS); choosing on it would be tuning on reported queries.
    assert at_095["param_value"] == 64 and at_095["recall_at_10"] == 0.94
    assert pick_operating_point(toy_rows(), 0.999) is None

## Results file
Rows are grouped by `(part, section)`. Rerunning a section replaces exactly its own rows, and the file is written atomically.

In [ ]:
def write_rows(rows, part, section, results_dir=None):
    path = Path(results_dir or RESULTS_DIR) / "results.csv"
    table = pd.DataFrame(rows).assign(part=part, section=section)
    if path.exists():
        old = pd.read_csv(path)
        mine = (old["part"].astype(str) == str(part)) & (old["section"].astype(str) == str(section))
        table = pd.concat([old[~mine], table], ignore_index=True)
    temporary = path.with_suffix(".tmp")
    table.to_csv(temporary, index=False)
    temporary.replace(path)
    return path


def read_results(results_dir=None):
    return pd.read_csv(Path(results_dir or RESULTS_DIR) / "results.csv")

In [ ]:
%%ipytest -qq

def test_write_rows_replaces_only_its_own_section(tmp_path):
    write_rows([dict(x=1), dict(x=2)], part=2, section="sift", results_dir=tmp_path)
    write_rows([dict(x=3)], part=2, section="glove", results_dir=tmp_path)
    write_rows([dict(x=4)], part=2, section="sift", results_dir=tmp_path)
    assert sorted(read_results(tmp_path)["x"]) == [3, 4]
    assert not list(tmp_path.glob("*.tmp"))

## Hypothesis guard
The Part 3 and Part 4 notebooks call `require_committed_hypothesis` first. It reads `hypothesis.md` as committed at `HEAD`, not the working copy, and refuses to run if that part's section is missing or still contains `_TODO_`.

In [ ]:
def hypothesis_section(text, part):
    """Body of the '## <part>...' section of hypothesis.md ('' if there is none)."""
    body, inside = [], False
    for line in text.splitlines():
        if line.startswith("## "):
            inside = line[3:].startswith(part)
        elif inside:
            body.append(line)
    return "\n".join(body).strip()


def hypothesis_ready(text, part):
    body = hypothesis_section(text, part)
    return bool(body) and "_TODO_" not in body


def require_committed_hypothesis(part):
    git = ["git", "-C", str(ROOT)]
    if subprocess.run(git + ["rev-parse", "--is-inside-work-tree"], capture_output=True).returncode != 0:
        print(f"warning: {ROOT} is not a git repository, so the {part} hypothesis commit cannot be checked")
        return
    committed = subprocess.run(git + ["show", "HEAD:hypothesis.md"], capture_output=True, text=True)
    if committed.returncode != 0 or not hypothesis_ready(committed.stdout, part):
        raise RuntimeError(f"The {part} hypothesis is not committed. Write it in hypothesis.md and commit it before running this experiment.")
    commit = subprocess.run(git + ["log", "-1", "--date=iso", "--format=%h %ad", "--", "hypothesis.md"],
                            capture_output=True, text=True).stdout.strip()
    print(f"{part} hypothesis committed in {commit}")

In [ ]:
%%ipytest -qq

SAMPLE = """# Hypotheses

## Part 3: Scaling
Exact search slows down linearly.

## Part 4: Hard geometry
_TODO_
"""


def test_sections_are_found():
    assert hypothesis_section(SAMPLE, "Part 3") == "Exact search slows down linearly."
    assert hypothesis_section(SAMPLE, "Part 4") == "_TODO_"
    assert hypothesis_section(SAMPLE, "Part 5") == ""


def test_only_filled_in_sections_are_ready():
    assert hypothesis_ready(SAMPLE, "Part 3")
    assert not hypothesis_ready(SAMPLE, "Part 4") and not hypothesis_ready(SAMPLE, "Part 5")

In [ ]:
print(f"mode={MODE}  results -> {RESULTS_DIR.relative_to(ROOT)}  timing CPU {TIMING_CPU}  build threads {BUILD_THREADS}")